# When one class is rare
**Machine Learning · Week 3 (Classification) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/ml/w3d3.html

When one class is rare, a classifier can get a tiny loss by mostly ignoring it.
Three fixes: class weights, resampling, and a different threshold.

In [ ]:
import numpy as np
import torch
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score, balanced_accuracy_score, accuracy_score

torch.set_num_threads(1)
X, y = make_classification(n_samples=3000, n_features=2, n_informative=2, n_redundant=0, n_clusters_per_class=1,
                           weights=[0.97, 0.03], class_sep=0.9, flip_y=0, random_state=2)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print(f"training set: {len(y_tr)} examples, {y_tr.sum()} rare ({y_tr.mean():.1%})")

def report(name, pred):
    print(f"{name:>22}: accuracy {accuracy_score(y_te, pred):.3f}  recall {recall_score(y_te, pred):.3f}  "
          f"precision {precision_score(y_te, pred, zero_division=0):.3f}  F1 {f1_score(y_te, pred):.3f}  "
          f"balanced {balanced_accuracy_score(y_te, pred):.3f}")

## Plain training versus class weights
`class_weight="balanced"` weighs every example by $n / (2\, n_{\text{class}})$, so both classes count equally in the loss.

In [ ]:
plain = LogisticRegression().fit(X_tr, y_tr)
weighted = LogisticRegression(class_weight="balanced").fit(X_tr, y_tr)
report("plain", plain.predict(X_te))
report("class weights", weighted.predict(X_te))

## The same in PyTorch
`BCEWithLogitsLoss(pos_weight=...)` multiplies the loss of every positive (rare) example.

In [ ]:
Xt = torch.tensor(X_tr, dtype=torch.float32)
yt = torch.tensor(y_tr, dtype=torch.float32).reshape(-1, 1)
pos_weight = torch.tensor([(y_tr == 0).sum() / (y_tr == 1).sum()])     # about 32: one rare example counts like 32 common ones
print(f"pos_weight = {pos_weight.item():.1f}")

torch.manual_seed(0)
model = torch.nn.Linear(2, 1)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
for step in range(500):
    optimizer.zero_grad()
    loss_fn(model(Xt), yt).backward()
    optimizer.step()
with torch.no_grad():
    pred = (model(torch.tensor(X_te, dtype=torch.float32)) > 0).int().numpy().ravel()
report("PyTorch, pos_weight", pred)

## Moving the threshold, and the precision–recall curve
Keep the plain model but change the cut-off. The **precision–recall curve** shows every threshold at once;
its area, the **average precision**, is a single score that cares about the rare class.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

score = plain.predict_proba(X_te)[:, 1]
for t in [0.5, 0.3, 0.1]:
    report(f"plain, threshold {t}", (score >= t).astype(int))

prec, rec, _ = precision_recall_curve(y_te, score)
plt.figure(figsize=(5.4, 3.8))
plt.plot(rec, prec, color="#ff8a50", lw=2.5, label=f"plain model (average precision {average_precision_score(y_te, score):.2f})")
plt.axhline(y_te.mean(), color="#bcb7cb", ls="--", label=f"random guessing ({y_te.mean():.2f})")
plt.xlabel("recall (rare examples found)")
plt.ylabel("precision (alarms that are right)")
plt.legend(loc="lower left")
plt.title("Every threshold, from strict (left) to generous (right)")
plt.show()

## Your turn: oversample by hand
Another fix is to **resample** the training set: repeat rare examples (with replacement) until both classes
are equally common, then train a plain model. Only ever resample the training set, never the test set.

In [ ]:
rng = np.random.default_rng(0)
rare_idx = np.flatnonzero(y_tr == 1)
common_idx = np.flatnonzero(y_tr == 0)
extra = np.array([], dtype=int)    # TODO: rng.choice(rare_idx, size=..., replace=True) to make the classes equal
idx = np.concatenate([common_idx, rare_idx, extra])
over = LogisticRegression().fit(X_tr[idx], y_tr[idx])
print("rare share after resampling:", round(y_tr[idx].mean(), 3))
report("oversampled", over.predict(X_te))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
extra = rng.choice(rare_idx, size=len(common_idx) - len(rare_idx), replace=True)
idx = np.concatenate([common_idx, rare_idx, extra])
over = LogisticRegression().fit(X_tr[idx], y_tr[idx])
print("rare share after resampling:", round(y_tr[idx].mean(), 3))
report("oversampled", over.predict(X_te))
report("class weights", weighted.predict(X_te))
print("Duplicating rare examples has almost the same effect as weighting them: both make the rare class count more.")